<a href="https://colab.research.google.com/github/fathimaaffu22-lab/AI-Battery-Project/blob/main/AI_Battery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!curl --max-time 5 -I http://localhost:8501

In [ ]:
import requests

try:
    response = requests.get(
        "https://loca.lt/mytunnelpassword",
        timeout=10
    )
    print(response.text)
except Exception as e:
    print("Password service unavailable:", e)


In [ ]:
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴your url is: https://full-radios-listen.loca.lt


In [ ]:
!streamlit run /content/drive/MyDrive/Batteryproject/app.py &>/content/logs.txt &

In [ ]:
!pip install streamlit -q

In [ ]:
%%writefile /content/drive/MyDrive/Batteryproject/app.py

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

# --------------------------------------------------
# PAGE SETUP
# --------------------------------------------------

st.set_page_config(
    page_title="BatteryGuard AI",
    page_icon="🔋",
    layout="wide"
)

st.title("🔋 BatteryGuard AI")
st.write(
    "AI-driven battery health, remaining useful life, "
    "uncertainty and anomaly monitoring."
)

# --------------------------------------------------
# LOAD DATA AND MODELS
# --------------------------------------------------

DATA_PATH = "/content/drive/MyDrive/Batteryproject/all_batteries_clean.csv"
MODEL_PATH = "/content/drive/MyDrive/Batteryproject/soh_random_forest.pkl"
ANOMALY_PATH = "/content/drive/MyDrive/Batteryproject/anomaly_model.pkl"
FEATURE_PATH = "/content/drive/MyDrive/Batteryproject/soh_features.pkl"

df = pd.read_csv(DATA_PATH)

soh_model = joblib.load(MODEL_PATH)
anomaly_model = joblib.load(ANOMALY_PATH)
features = joblib.load(FEATURE_PATH)

# Features used by anomaly model
anomaly_features = [
    'Avg_Voltage_V',
    'Min_Voltage_V',
    'Max_Voltage_V',
    'Avg_Current_A',
    'Min_Current_A',
    'Max_Current_A',
    'Avg_Battery_Temp_C',
    'Max_Battery_Temp_C',
    'Temp_Rise_C',
    'Discharge_Duration_s'
]

# --------------------------------------------------
# BATTERY SELECTOR
# --------------------------------------------------

battery_id = st.sidebar.selectbox(
    "Select Battery",
    sorted(df["Battery_ID"].unique())
)

battery_df = df[
    df["Battery_ID"] == battery_id
].copy()

cycle = st.sidebar.slider(
    "Select Discharge Cycle",
    min_value=int(battery_df["Discharge_Cycle"].min()),
    max_value=int(battery_df["Discharge_Cycle"].max()),
    value=int(battery_df["Discharge_Cycle"].max())
)

history = battery_df[
    battery_df["Discharge_Cycle"] <= cycle
].copy()

current_row = history.iloc[-1]

# --------------------------------------------------
# SOH PREDICTION
# --------------------------------------------------

X_current = current_row[features].to_frame().T

predicted_soh = soh_model.predict(X_current)[0]

# Prediction uncertainty from individual trees
tree_predictions = np.array([
    tree.predict(X_current)[0]
    for tree in soh_model.estimators_
])

lower_soh = np.percentile(tree_predictions, 5)
upper_soh = np.percentile(tree_predictions, 95)

# --------------------------------------------------
# RUL ESTIMATION
# --------------------------------------------------

# Generate AI SOH predictions for history
X_history = history[features]

history["Predicted_SOH"] = soh_model.predict(X_history)

# Fit simple degradation trend
if len(history) >= 5:

    x = history["Discharge_Cycle"].values
    y = history["Predicted_SOH"].values

    slope, intercept = np.polyfit(x, y, 1)

    if slope < 0:
        predicted_eol = (80 - intercept) / slope
        predicted_rul = max(predicted_eol - cycle, 0)
    else:
        predicted_eol = np.nan
        predicted_rul = np.nan

else:
    predicted_eol = np.nan
    predicted_rul = np.nan

# --------------------------------------------------
# ANOMALY DETECTION
# --------------------------------------------------

X_anomaly = current_row[anomaly_features].to_frame().T

anomaly_prediction = anomaly_model.predict(X_anomaly)[0]

ml_anomaly = anomaly_prediction == -1

# Dataset-relative operational warning rules
temp_threshold = df["Max_Battery_Temp_C"].quantile(0.95)
temp_rise_threshold = df["Temp_Rise_C"].quantile(0.95)

rule_alert = (
    current_row["Max_Battery_Temp_C"] > temp_threshold
    or
    current_row["Temp_Rise_C"] > temp_rise_threshold
)

overall_alert = ml_anomaly or rule_alert

# --------------------------------------------------
# TOP METRICS
# --------------------------------------------------

col1, col2, col3, col4 = st.columns(4)

col1.metric(
    "AI Predicted SOH",
    f"{predicted_soh:.2f}%"
)

col2.metric(
    "Uncertainty Range",
    f"{lower_soh:.1f}% – {upper_soh:.1f}%"
)

if not np.isnan(predicted_rul):
    col3.metric(
        "Predicted RUL",
        f"{predicted_rul:.1f} cycles"
    )
else:
    col3.metric(
        "Predicted RUL",
        "Not enough trend"
    )

if overall_alert:
    col4.metric(
        "Battery Status",
        "⚠️ ALERT"
    )
else:
    col4.metric(
        "Battery Status",
        "✅ NORMAL"
    )

st.divider()

# --------------------------------------------------
# SOH HISTORY GRAPH
# --------------------------------------------------

st.subheader("SOH Degradation & AI Prediction")

graph_history = history.copy()

all_tree_history = np.array([
    tree.predict(X_history)
    for tree in soh_model.estimators_
])

lower_history = np.percentile(
    all_tree_history,
    5,
    axis=0
)

upper_history = np.percentile(
    all_tree_history,
    95,
    axis=0
)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    graph_history["Discharge_Cycle"],
    graph_history["SOH_Percent"],
    label="Actual SOH"
)

ax.plot(
    graph_history["Discharge_Cycle"],
    graph_history["Predicted_SOH"],
    label="AI Predicted SOH"
)

ax.fill_between(
    graph_history["Discharge_Cycle"],
    lower_history,
    upper_history,
    alpha=0.2,
    label="Prediction Uncertainty"
)

ax.axhline(
    80,
    linestyle="--",
    label="EOL Threshold (80%)"
)

ax.set_xlabel("Discharge Cycle")
ax.set_ylabel("SOH (%)")
ax.legend()
ax.grid(True)

st.pyplot(fig)

# --------------------------------------------------
# CURRENT SENSOR INFORMATION
# --------------------------------------------------

st.subheader("Current Battery Signals")

c1, c2, c3, c4 = st.columns(4)

c1.metric(
    "Average Voltage",
    f"{current_row['Avg_Voltage_V']:.3f} V"
)

c2.metric(
    "Average Current",
    f"{current_row['Avg_Current_A']:.3f} A"
)

c3.metric(
    "Max Temperature",
    f"{current_row['Max_Battery_Temp_C']:.2f} °C"
)

c4.metric(
    "Discharge Duration",
    f"{current_row['Discharge_Duration_s']:.0f} s"
)

# --------------------------------------------------
# ALERT DETAILS
# --------------------------------------------------

st.subheader("Safety & Anomaly Monitoring")

if ml_anomaly:
    st.warning(
        "Isolation Forest detected unusual battery behaviour."
    )
else:
    st.success(
        "No ML anomaly detected for the selected cycle."
    )

if rule_alert:
    st.warning(
        "Operational temperature warning threshold triggered."
    )
else:
    st.success(
        "Temperature behaviour is within the POC operating range."
    )

# --------------------------------------------------
# FEATURE IMPORTANCE
# --------------------------------------------------

st.subheader("SOH Model Explanation")

st.write(
    "This chart shows which battery signals the "
    "Random Forest relied on most when predicting SOH."
)

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": soh_model.feature_importances_
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)

st.bar_chart(
    importance_df.set_index("Feature"),
    horizontal=True
)

# --------------------------------------------------
# MODEL PERFORMANCE
# --------------------------------------------------

st.subheader("SOH Model Performance — Unseen Battery B0018")

st.caption(
    "These evaluation results were calculated on B0018, "
    "which was not used to train the SOH model. "
    "They do not change when another battery is selected."
)

In [ ]:
joblib.dump(
    features,
    '/content/drive/MyDrive/Batteryproject/soh_features.pkl'
)

In [ ]:
import joblib

# Save cleaned dataset
all_batteries_df.to_csv(
    '/content/drive/MyDrive/Batteryproject/all_batteries_clean.csv',
    index=False
)

# Save SOH Random Forest model
joblib.dump(
    rf_model,
    '/content/drive/MyDrive/Batteryproject/soh_random_forest.pkl'
)

# Save anomaly model
joblib.dump(
    anomaly_model,
    '/content/drive/MyDrive/Batteryproject/anomaly_model.pkl'
)

print("Files saved successfully!")


In [ ]:
shap.initjs()

shap.force_plot(
    explainer.expected_value,
    shap_values[0],
    X_test.iloc[0]
)

In [ ]:
shap.summary_plot(
    shap_values,
    X_test
)

In [ ]:
import shap

explainer = shap.TreeExplainer(rf_model)

shap_values = explainer.shap_values(X_test)

In [ ]:
!pip install shap -q

In [ ]:
all_batteries_df['Overall_Alert'] = np.where(
    (all_batteries_df['Anomaly_Flag'] == -1) |
    (all_batteries_df['Safety_Alert'] == True),
    'Alert',
    'Normal'
)

print(all_batteries_df['Overall_Alert'].value_counts())

In [ ]:
all_batteries_df['Safety_Alert'] = (
    (all_batteries_df['Max_Battery_Temp_C'] > temp_threshold) |
    (all_batteries_df['Temp_Rise_C'] > temp_rise_threshold)
)

print(all_batteries_df['Safety_Alert'].value_counts())

In [ ]:
temp_threshold = all_batteries_df['Max_Battery_Temp_C'].quantile(0.95)
temp_rise_threshold = all_batteries_df['Temp_Rise_C'].quantile(0.95)

print("95th percentile max temperature:", round(temp_threshold, 2))
print("95th percentile temperature rise:", round(temp_rise_threshold, 2))

In [ ]:
anomalies = all_batteries_df[
    all_batteries_df['Anomaly_Flag'] == -1
]

print(
    anomalies[
        [
            'Battery_ID',
            'Discharge_Cycle',
            'Avg_Voltage_V',
            'Avg_Current_A',
            'Max_Battery_Temp_C',
            'Temp_Rise_C',
            'Discharge_Duration_s'
        ]
    ].head(20)
)

In [ ]:
print(all_batteries_df['Anomaly_Flag'].value_counts())

In [ ]:
anomaly_model = IsolationForest(
    contamination=0.03,
    random_state=42
)

all_batteries_df['Anomaly_Flag'] = anomaly_model.fit_predict(X_anomaly)

In [ ]:
from sklearn.ensemble import IsolationForest

anomaly_features = [
    'Avg_Voltage_V',
    'Min_Voltage_V',
    'Max_Voltage_V',
    'Avg_Current_A',
    'Min_Current_A',
    'Max_Current_A',
    'Avg_Battery_Temp_C',
    'Max_Battery_Temp_C',
    'Temp_Rise_C',
    'Discharge_Duration_s'
]

X_anomaly = all_batteries_df[anomaly_features]

In [ ]:
pred_eol, pred_rul = predict_rul_from_soh_trend(
    rul_18,
    current_cycle=50
)

print("Predicted EOL:", round(pred_eol, 1))
print("Predicted RUL:", round(pred_rul, 1), "cycles")

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

def predict_rul_from_soh_trend(df, current_cycle, eol_threshold=80):
    history = df[df['Discharge_Cycle'] <= current_cycle].copy()

    X = history[['Discharge_Cycle']]
    y = history['SOH_Percent']

    model = LinearRegression()
    model.fit(X, y)

    slope = model.coef_[0]
    intercept = model.intercept_

    predicted_eol_cycle = (eol_threshold - intercept) / slope
    predicted_rul = predicted_eol_cycle - current_cycle

    return predicted_eol_cycle, max(predicted_rul, 0)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rul_mae = mean_absolute_error(
    y_rul_test,
    rul_predictions
)

rul_rmse = np.sqrt(
    mean_squared_error(y_rul_test, rul_predictions)
)

rul_r2 = r2_score(
    y_rul_test,
    rul_predictions
)

print("Random Forest RUL Results")
print("-------------------------")
print("MAE :", round(rul_mae, 2), "cycles")
print("RMSE:", round(rul_rmse, 2), "cycles")
print("R²  :", round(rul_r2, 3))

In [ ]:
rul_predictions = rul_model.predict(X_rul_test)

print("First 10 predicted RUL values:")
print(rul_predictions[:10])

print("\nFirst 10 actual RUL values:")
print(y_rul_test.head(10).values)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rul_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rul_model.fit(X_rul_train, y_rul_train)

print("RUL model training completed!")

In [ ]:
rul_features = [
    'Discharge_Cycle',
    'Avg_Voltage_V',
    'Min_Voltage_V',
    'Max_Voltage_V',
    'Avg_Current_A',
    'Min_Current_A',
    'Max_Current_A',
    'Avg_Battery_Temp_C',
    'Max_Battery_Temp_C',
    'Temp_Rise_C',
    'Discharge_Duration_s'
]

X_rul_train = rul_train_df[rul_features]
y_rul_train = rul_train_df['RUL_Cycles']

X_rul_test = rul_test_df[rul_features]
y_rul_test = rul_test_df['RUL_Cycles']

In [ ]:
rul_train_df = pd.concat([
    rul_5[rul_5['Discharge_Cycle'] <= 101],
    rul_6[rul_6['Discharge_Cycle'] <= 61],
    rul_7[rul_7['Discharge_Cycle'] <= 124]
], ignore_index=True)

rul_test_df = rul_18[
    rul_18['Discharge_Cycle'] <= 75
].copy()

print("RUL training rows:", len(rul_train_df))
print("RUL testing rows:", len(rul_test_df))

In [ ]:
for battery_id, df in [
    ('B0005', rul_5),
    ('B0006', rul_6),
    ('B0007', rul_7),
    ('B0018', rul_18)
]:

    eol_row = df[df['RUL_Cycles'] == 0].iloc[0]

    print(
        battery_id,
        "EOL cycle:",
        eol_row['Discharge_Cycle'],
        "| SOH:",
        eol_row['SOH_Percent']
    )

In [ ]:
print(
    rul_18[
        ['Discharge_Cycle', 'SOH_Percent', 'RUL_Cycles']
    ].iloc[45:77]
)

In [ ]:
rul_5 = add_rul_target(df_5)
rul_6 = add_rul_target(df_6)
rul_7 = add_rul_target(df_7)
rul_18 = add_rul_target(df_18)

In [ ]:
def add_rul_target(df, eol_threshold=80):
    df = df.copy()

    # Find first cycle where SOH reaches or drops below EOL threshold
    eol_rows = df[df['SOH_Percent'] <= eol_threshold]

    if len(eol_rows) == 0:
        return None

    eol_cycle = eol_rows['Discharge_Cycle'].iloc[0]

    # Remaining Useful Life
    df['RUL_Cycles'] = eol_cycle - df['Discharge_Cycle']

    # After EOL, RUL should not be negative
    df['RUL_Cycles'] = df['RUL_Cycles'].clip(lower=0)

    return df

In [ ]:
predicted_eol_cycle = (80 - intercept) / slope
predicted_rul = predicted_eol_cycle - current_cycle

print("Predicted EOL cycle:", round(predicted_eol_cycle, 1))
print("Predicted RUL:", round(predicted_rul, 1), "cycles")

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

rul_baseline = LinearRegression()

X_history = history[['Discharge_Cycle']]
y_history = history['SOH_Percent']

rul_baseline.fit(X_history, y_history)

slope = rul_baseline.coef_[0]
intercept = rul_baseline.intercept_

print("Slope:", slope)
print("Intercept:", intercept)


In [ ]:
current_cycle = 50

history = b18[
    b18['Discharge_Cycle'] <= current_cycle
].copy()

print(history[
    ['Discharge_Cycle', 'SOH_Percent']
].tail())

In [ ]:
eol_threshold = 80

b18 = test_df.copy()

below_eol = b18[b18['SOH_Percent'] <= eol_threshold]

print(below_eol[
    ['Discharge_Cycle', 'SOH_Percent']
].head())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

# Actual SOH
plt.plot(
    test_df['Discharge_Cycle'],
    y_test,
    label='Actual SOH'
)

# Predicted SOH
plt.plot(
    test_df['Discharge_Cycle'],
    rf_mean_prediction,
    label='Predicted SOH'
)

# Uncertainty band
plt.fill_between(
    test_df['Discharge_Cycle'],
    rf_lower,
    rf_upper,
    alpha=0.2,
    label='Prediction Uncertainty Range'
)

plt.xlabel('Discharge Cycle')
plt.ylabel('SOH (%)')
plt.title('SOH Prediction with Uncertainty - B0018')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
import numpy as np

# Get prediction from every tree in the Random Forest
all_tree_predictions = np.array([
    tree.predict(X_test) for tree in rf_model.estimators_
])

# Average prediction across trees
rf_mean_prediction = all_tree_predictions.mean(axis=0)

# Lower and upper bounds using percentiles
rf_lower = np.percentile(all_tree_predictions, 5, axis=0)
rf_upper = np.percentile(all_tree_predictions, 95, axis=0)

print("First 5 predictions with uncertainty:\n")

for i in range(5):
    print(
        f"Cycle {test_df['Discharge_Cycle'].iloc[i]}: "
        f"Prediction = {rf_mean_prediction[i]:.2f}% | "
        f"Range = {rf_lower[i]:.2f}% to {rf_upper[i]:.2f}%"
    )

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.barh(
    feature_importance['Feature'],
    feature_importance['Importance']
)

plt.xlabel('Feature Importance')
plt.ylabel('Feature')
plt.title('Random Forest Feature Importance')

plt.gca().invert_yaxis()
plt.show()

In [ ]:
import pandas as pd

feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

print(feature_importance)

In [ ]:
mae_no_duration = mean_absolute_error(
    y_test,
    pred_no_duration
)

rmse_no_duration = np.sqrt(
    mean_squared_error(y_test, pred_no_duration)
)

r2_no_duration = r2_score(
    y_test,
    pred_no_duration
)

print("Random Forest WITHOUT Duration")
print("------------------------------")
print("MAE :", round(mae_no_duration, 2))
print("RMSE:", round(rmse_no_duration, 2))
print("R²  :", round(r2_no_duration, 3))

In [ ]:
rf_no_duration = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf_no_duration.fit(
    X_train_no_duration,
    y_train
)

pred_no_duration = rf_no_duration.predict(
    X_test_no_duration
)

In [ ]:
X_train_no_duration = train_df[features_no_duration]
X_test_no_duration = test_df[features_no_duration]

In [ ]:
features_no_duration = [
    'Avg_Voltage_V',
    'Min_Voltage_V',
    'Max_Voltage_V',
    'Avg_Current_A',
    'Min_Current_A',
    'Max_Current_A',
    'Avg_Battery_Temp_C',
    'Max_Battery_Temp_C',
    'Temp_Rise_C'
]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.plot(
    test_df['Discharge_Cycle'],
    y_test,
    label='Actual SOH'
)

plt.plot(
    test_df['Discharge_Cycle'],
    rf_predictions,
    label='Predicted SOH'
)

plt.xlabel('Discharge Cycle')
plt.ylabel('SOH (%)')
plt.title('Actual vs Predicted SOH - B0018')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rf_mae = mean_absolute_error(y_test, rf_predictions)

rf_rmse = np.sqrt(
    mean_squared_error(y_test, rf_predictions)
)

rf_r2 = r2_score(y_test, rf_predictions)

print("Random Forest Results")
print("---------------------")
print("MAE :", round(rf_mae, 2))
print("RMSE:", round(rf_rmse, 2))
print("R²  :", round(rf_r2, 3))

In [ ]:
rf_predictions = rf_model.predict(X_test)

print("First 10 predicted SOH values:")
print(rf_predictions[:10])

print("\nFirst 10 actual SOH values:")
print(y_test.head(10).values)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Create the Random Forest
rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

# TRAIN the model
rf_model.fit(X_train, y_train)

print("Random Forest training completed!")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Baseline uses ONLY discharge cycle
X_train_baseline = train_df[['Discharge_Cycle']]
X_test_baseline = test_df[['Discharge_Cycle']]

# Create and train baseline
baseline_model = LinearRegression()

baseline_model.fit(
    X_train_baseline,
    y_train
)

# Predict SOH of unseen B0018
baseline_predictions = baseline_model.predict(X_test_baseline)

# Measure performance
baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(y_test, baseline_predictions)
)

baseline_r2 = r2_score(
    y_test,
    baseline_predictions
)

print("Baseline Results")
print("----------------")
print("MAE :", round(baseline_mae, 2))
print("RMSE:", round(baseline_rmse, 2))
print("R²  :", round(baseline_r2, 3))

In [ ]:
# Features we will give to the AI
features = [
    'Avg_Voltage_V',
    'Min_Voltage_V',
    'Max_Voltage_V',
    'Avg_Current_A',
    'Min_Current_A',
    'Max_Current_A',
    'Avg_Battery_Temp_C',
    'Max_Battery_Temp_C',
    'Temp_Rise_C',
    'Discharge_Duration_s'
]

# Training batteries
train_df = all_batteries_df[
    all_batteries_df['Battery_ID'].isin(
        ['B0005', 'B0006', 'B0007']
    )
].copy()

# Completely unseen test battery
test_df = all_batteries_df[
    all_batteries_df['Battery_ID'] == 'B0018'
].copy()


# X = information given to AI
X_train = train_df[features]
X_test = test_df[features]

# y = correct answer
y_train = train_df['SOH_Percent']
y_test = test_df['SOH_Percent']


print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nNumber of input features:", len(features))
print("Input features:")
print(features)

In [ ]:
import matplotlib.pyplot as plt

for battery_id in all_batteries_df['Battery_ID'].unique():

    battery_data = all_batteries_df[
        all_batteries_df['Battery_ID'] == battery_id
    ]

    plt.plot(
        battery_data['Discharge_Cycle'],
        battery_data['SOH_Percent'],
        label=battery_id
    )

plt.xlabel('Discharge Cycle')
plt.ylabel('SOH (%)')
plt.title('SOH Degradation of NASA Batteries')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
print(all_batteries_df['Battery_ID'].value_counts())

print("\nMissing values:")
print(all_batteries_df.isna().sum())

In [ ]:
all_batteries_df = pd.concat(
    [df_5, df_6, df_7, df_18],
    ignore_index=True
)

print("Combined dataset shape:", all_batteries_df.shape)

all_batteries_df.head(10)

In [ ]:
df_5 = extract_battery("B0005")
df_6 = extract_battery("B0006")
df_7 = extract_battery("B0007")
df_18 = extract_battery("B0018")

print("B0005 rows:", len(df_5))
print("B0006 rows:", len(df_6))
print("B0007 rows:", len(df_7))
print("B0018 rows:", len(df_18))

In [ ]:
import scipy.io
import pandas as pd
import numpy as np

def extract_battery(battery_id):

    file_path = f'/content/drive/MyDrive/Batteryproject/{battery_id}.mat'

    # Load .mat file
    mat_data = scipy.io.loadmat(file_path)

    battery_struct = mat_data[battery_id][0, 0]
    cycles = battery_struct['cycle'][0]

    summary_list = []

    # Helper to extract FULL arrays correctly
    def get_array(measurements, field_name):
        if field_name in measurements.dtype.names:
            return np.asarray(
                measurements[field_name]
            ).flatten().astype(float)
        else:
            return np.array([])

    # Go through every NASA operation
    for idx, cycle in enumerate(cycles):

        op_type = cycle['type'][0]
        ambient_temp = cycle['ambient_temperature'][0, 0]
        original_operation_index = idx + 1

        measurements = cycle['data'][0, 0]

        # For now, only discharge operations
        if op_type == 'discharge':

            # Capacity
            if 'Capacity' in measurements.dtype.names:
                capacity_array = np.asarray(
                    measurements['Capacity']
                ).flatten()

                capacity = float(capacity_array[0])
            else:
                capacity = np.nan

            # Full measurement arrays
            voltage = get_array(
                measurements,
                'Voltage_measured'
            )

            current = get_array(
                measurements,
                'Current_measured'
            )

            battery_temp = get_array(
                measurements,
                'Temperature_measured'
            )

            time = get_array(
                measurements,
                'Time'
            )

            # Voltage features
            avg_voltage = np.mean(voltage) if len(voltage) > 0 else np.nan
            min_voltage = np.min(voltage) if len(voltage) > 0 else np.nan
            max_voltage = np.max(voltage) if len(voltage) > 0 else np.nan

            # Current features
            avg_current = np.mean(current) if len(current) > 0 else np.nan
            min_current = np.min(current) if len(current) > 0 else np.nan
            max_current = np.max(current) if len(current) > 0 else np.nan

            # Temperature features
            avg_battery_temp = np.mean(battery_temp) if len(battery_temp) > 0 else np.nan
            max_battery_temp = np.max(battery_temp) if len(battery_temp) > 0 else np.nan

            temp_rise = (
                battery_temp[-1] - battery_temp[0]
                if len(battery_temp) > 1
                else np.nan
            )

            # Discharge duration
            discharge_duration = (
                time[-1] - time[0]
                if len(time) > 1
                else np.nan
            )

            summary_list.append({

                'Battery_ID': battery_id,

                'Original_Operation_Index':
                    original_operation_index,

                'Operation':
                    op_type,

                'Ambient_Temp_C':
                    ambient_temp,

                'Capacity_Ah':
                    capacity,

                'Avg_Voltage_V':
                    avg_voltage,

                'Min_Voltage_V':
                    min_voltage,

                'Max_Voltage_V':
                    max_voltage,

                'Avg_Current_A':
                    avg_current,

                'Min_Current_A':
                    min_current,

                'Max_Current_A':
                    max_current,

                'Avg_Battery_Temp_C':
                    avg_battery_temp,

                'Max_Battery_Temp_C':
                    max_battery_temp,

                'Temp_Rise_C':
                    temp_rise,

                'Discharge_Duration_s':
                    discharge_duration
            })

    # Create dataframe
    df = pd.DataFrame(summary_list)

    # Add clean discharge cycle number
    df.insert(
        1,
        'Discharge_Cycle',
        range(1, len(df) + 1)
    )

    # SOH based on first measured discharge capacity
    initial_capacity = df['Capacity_Ah'].iloc[0]

    df['SOH_Percent'] = (
        df['Capacity_Ah'] / initial_capacity
    ) * 100

    # Round values
    df['Capacity_Ah'] = df['Capacity_Ah'].round(4)
    df['SOH_Percent'] = df['SOH_Percent'].round(2)

    df['Avg_Voltage_V'] = df['Avg_Voltage_V'].round(3)
    df['Min_Voltage_V'] = df['Min_Voltage_V'].round(3)
    df['Max_Voltage_V'] = df['Max_Voltage_V'].round(3)

    df['Avg_Current_A'] = df['Avg_Current_A'].round(3)
    df['Min_Current_A'] = df['Min_Current_A'].round(3)
    df['Max_Current_A'] = df['Max_Current_A'].round(3)

    df['Avg_Battery_Temp_C'] = df['Avg_Battery_Temp_C'].round(2)
    df['Max_Battery_Temp_C'] = df['Max_Battery_Temp_C'].round(2)
    df['Temp_Rise_C'] = df['Temp_Rise_C'].round(2)

    df['Discharge_Duration_s'] = df['Discharge_Duration_s'].round(1)

    return df

In [ ]:
# Diagnostic check: see how NASA stores the discharge arrays

test_cycle = None

for cycle in cycles:
    if cycle['type'][0] == 'discharge':
        test_cycle = cycle
        break

measurements = test_cycle['data'][0, 0]

print("Available fields:")
print(measurements.dtype.names)

print("\nShapes of important fields:")

for field in [
    'Voltage_measured',
    'Current_measured',
    'Temperature_measured',
    'Time'
]:
    if field in measurements.dtype.names:
        arr = measurements[field]
        print(field, "->", np.shape(arr))

        try:
            inner = measurements[field][0, 0]
            print("   inner shape:", np.shape(inner))
            print("   first few values:", np.asarray(inner).flatten()[:5])
        except Exception as e:
            print("   Could not inspect inner data:", e)


In [ ]:
# Save the complete cleaned B0005 table as an Excel file

excel_path = '/content/drive/MyDrive/Batteryproject/B0005_Cleaned_Data.xlsx'

df.to_excel(excel_path, index=False)

print("Excel file saved successfully!")
print(excel_path)

In [ ]:
# Check how many rows and columns are in our table
print(f"Your dataset contains exactly {df.shape[0]} rows of data!")


In [ ]:
import matplotlib.pyplot as plt

# 1. Choose the size and style of your graph window
plt.figure(figsize=(10, 6))

# 2. Plot Cycle_Index on the bottom (X-axis) and Capacity on the side (Y-axis)
# 'marker=o' adds small dots on each data point, 'color=red' makes it stand out
plt.plot(df['Cycle_Index'], df['Capacity_Ah'], marker='o', color='red', linestyle='-', markersize=4)

# 3. Add clear titles and labels so anyone can understand your chart
plt.title('NASA Battery B0005: Capacity Degradation Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Battery Test Cycle Number', fontsize=12)
plt.ylabel('Battery Capacity (Ah)', fontsize=12)

# 4. Turn on a grid background and render the final chart
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


In [ ]:
import scipy.io
import pandas as pd
import numpy as np

# 1. File path
file_path = '/content/drive/MyDrive/Batteryproject/B0005.mat'

# 2. Load NASA MATLAB file
mat_data = scipy.io.loadmat(file_path)

battery_struct = mat_data['B0005'][0, 0]
cycles = battery_struct['cycle'][0]

summary_list = []

# Helper function to correctly extract the FULL measurement array
def get_array(measurements, field_name):
    if field_name in measurements.dtype.names:
        return np.asarray(
            measurements[field_name]
        ).flatten().astype(float)
    else:
        return np.array([])


# 3. Go through every NASA operation
for idx, cycle in enumerate(cycles):

    op_type = cycle['type'][0]
    ambient_temp = cycle['ambient_temperature'][0, 0]
    original_operation_index = idx + 1

    measurements = cycle['data'][0, 0]

    # Only discharge operations for our SOH table
    if op_type == 'discharge':

        # -----------------------
        # CAPACITY
        # -----------------------
        if 'Capacity' in measurements.dtype.names:
            capacity_array = np.asarray(
                measurements['Capacity']
            ).flatten()

            capacity = float(capacity_array[0])
        else:
            capacity = np.nan


        # -----------------------
        # FULL TIME-SERIES ARRAYS
        # -----------------------
        voltage = get_array(
            measurements,
            'Voltage_measured'
        )

        current = get_array(
            measurements,
            'Current_measured'
        )

        battery_temp = get_array(
            measurements,
            'Temperature_measured'
        )

        time = get_array(
            measurements,
            'Time'
        )


        # -----------------------
        # VOLTAGE FEATURES
        # -----------------------
        avg_voltage = (
            np.mean(voltage)
            if len(voltage) > 0
            else np.nan
        )

        min_voltage = (
            np.min(voltage)
            if len(voltage) > 0
            else np.nan
        )

        max_voltage = (
            np.max(voltage)
            if len(voltage) > 0
            else np.nan
        )


        # -----------------------
        # CURRENT FEATURES
        # -----------------------
        avg_current = (
            np.mean(current)
            if len(current) > 0
            else np.nan
        )

        min_current = (
            np.min(current)
            if len(current) > 0
            else np.nan
        )

        max_current = (
            np.max(current)
            if len(current) > 0
            else np.nan
        )


        # -----------------------
        # TEMPERATURE FEATURES
        # -----------------------
        avg_battery_temp = (
            np.mean(battery_temp)
            if len(battery_temp) > 0
            else np.nan
        )

        max_battery_temp = (
            np.max(battery_temp)
            if len(battery_temp) > 0
            else np.nan
        )

        if len(battery_temp) > 1:
            temp_rise = (
                battery_temp[-1]
                - battery_temp[0]
            )
        else:
            temp_rise = np.nan


        # -----------------------
        # DISCHARGE DURATION
        # -----------------------
        if len(time) > 1:
            discharge_duration = (
                time[-1] - time[0]
            )
        else:
            discharge_duration = np.nan


        # Store one summary row for this discharge cycle
        summary_list.append({

            'Original_Operation_Index':
                original_operation_index,

            'Operation':
                op_type,

            'Ambient_Temp_C':
                ambient_temp,

            'Capacity_Ah':
                capacity,

            'Avg_Voltage_V':
                avg_voltage,

            'Min_Voltage_V':
                min_voltage,

            'Max_Voltage_V':
                max_voltage,

            'Avg_Current_A':
                avg_current,

            'Min_Current_A':
                min_current,

            'Max_Current_A':
                max_current,

            'Avg_Battery_Temp_C':
                avg_battery_temp,

            'Max_Battery_Temp_C':
                max_battery_temp,

            'Temp_Rise_C':
                temp_rise,

            'Discharge_Duration_s':
                discharge_duration
        })


# 4. Create DataFrame
df = pd.DataFrame(summary_list)


# 5. Add our clean discharge-cycle number
df.insert(
    0,
    'Discharge_Cycle',
    range(1, len(df) + 1)
)


# 6. Calculate SOH
initial_capacity = df['Capacity_Ah'].iloc[0]

df['SOH_Percent'] = (
    df['Capacity_Ah']
    / initial_capacity
) * 100


# 7. Round for clean display
df['Capacity_Ah'] = \
    df['Capacity_Ah'].round(4)

df['SOH_Percent'] = \
    df['SOH_Percent'].round(2)

df['Avg_Voltage_V'] = \
    df['Avg_Voltage_V'].round(3)

df['Min_Voltage_V'] = \
    df['Min_Voltage_V'].round(3)

df['Max_Voltage_V'] = \
    df['Max_Voltage_V'].round(3)

df['Avg_Current_A'] = \
    df['Avg_Current_A'].round(3)

df['Min_Current_A'] = \
    df['Min_Current_A'].round(3)

df['Max_Current_A'] = \
    df['Max_Current_A'].round(3)

df['Avg_Battery_Temp_C'] = \
    df['Avg_Battery_Temp_C'].round(2)

df['Max_Battery_Temp_C'] = \
    df['Max_Battery_Temp_C'].round(2)

df['Temp_Rise_C'] = \
    df['Temp_Rise_C'].round(2)

df['Discharge_Duration_s'] = \
    df['Discharge_Duration_s'].round(1)


print(
    "--- CORRECTED B0005 BATTERY DATA ---"
)

df.head(10)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')